# Célula de configurações

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col, trim, lower, regexp_replace

os.environ["AWS_REGION"] = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"

iceberg_version = "1.4.3"
aws_version = "3.3.4"

packages = [
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{iceberg_version}",
    f"org.apache.iceberg:iceberg-aws-bundle:{iceberg_version}",
    f"org.apache.hadoop:hadoop-aws:{aws_version}",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

spark = SparkSession.builder \
    .appName("Ingestao-Lakehouse-CGE") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hive") \
    .config("spark.sql.catalog.iceberg.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.iceberg.client.region", "us-east-1") \
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


print("Spark configurado e conectado ao Iceberg/MinIO!")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-198ba862-4516-4498-9fbf-c9e18189cffb;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.3 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.4.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.4.3/iceberg-spark-runtime-3.5_2.12-1.4.3.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12

Spark configurado e conectado ao Iceberg/MinIO!


# Criação de catalogo e banco de dados no Iceberg

In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.silver")
print("Namespace 'iceberg.silver' verificado/criado.")

Namespace 'iceberg.silver' verificado/criado.


# Leitura do arquivo de ingestão

In [5]:
file_path = "/workspaces/lakehouse-lab/data/dados_raw.csv"

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .csv(file_path)

print(f"Total de registros lidos: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

Total de registros lidos: 5
root
 |-- id: integer (nullable = true)
 |-- data_relatorio: date (nullable = true)
 |-- departamento: string (nullable = true)
 |-- assunto: string (nullable = true)
 |-- status: string (nullable = true)
 |-- responsavel: string (nullable = true)
 |-- horas_trabalhadas: integer (nullable = true)
 |-- valor_investido: double (nullable = true)

+---+--------------+------------+---------------------+------------+--------------+-----------------+---------------+
|id |data_relatorio|departamento|assunto              |status      |responsavel   |horas_trabalhadas|valor_investido|
+---+--------------+------------+---------------------+------------+--------------+-----------------+---------------+
|1  |2024-01-15    |TI          |Implementação Sistema|Concluído   |João Silva    |40               |5000.0         |
|2  |2024-01-16    |RH          |Recrutamento Q1      |Em Progresso|Maria Santos  |20               |2500.0         |
|3  |2024-01-17    |Vendas      |Aná

# Transformação e limpeza dos dados 

In [6]:
def clean_column_names(df):
    for col_name in df.columns:
        clean_name = col_name.strip().replace(" ", "_").lower()
        df = df.withColumnRenamed(col_name, clean_name)
    return df

df_clean = clean_column_names(df_raw)

df_silver = df_clean.withColumn("data_ingestao", current_timestamp())

print("Schema após o refinamento:")
df_silver.printSchema()

Schema após o refinamento:
root
 |-- id: integer (nullable = true)
 |-- data_relatorio: date (nullable = true)
 |-- departamento: string (nullable = true)
 |-- assunto: string (nullable = true)
 |-- status: string (nullable = true)
 |-- responsavel: string (nullable = true)
 |-- horas_trabalhadas: integer (nullable = true)
 |-- valor_investido: double (nullable = true)
 |-- data_ingestao: timestamp (nullable = false)



# Migração dos dados 

In [7]:
table_name = "iceberg.silver.dados_auditoria"

df_silver.writeTo(table_name) \
    .tableProperty("format-version", "2") \
    .using("iceberg") \
    .createOrReplace()

print(f"Tabela {table_name} gravada com sucesso no MinIO/Iceberg!")

Tabela iceberg.silver.dados_auditoria gravada com sucesso no MinIO/Iceberg!


# Vizualização dos dados escritos

In [8]:
spark.sql(f"SELECT * FROM {table_name} LIMIT 5").show()

spark.sql(f"SELECT * FROM {table_name}.snapshots").show(truncate=False)

+---+--------------+------------+--------------------+------------+--------------+-----------------+---------------+--------------------+
| id|data_relatorio|departamento|             assunto|      status|   responsavel|horas_trabalhadas|valor_investido|       data_ingestao|
+---+--------------+------------+--------------------+------------+--------------+-----------------+---------------+--------------------+
|  1|    2024-01-15|          TI|Implementação Sis...|   Concluído|    João Silva|               40|         5000.0|2026-04-22 12:12:...|
|  2|    2024-01-16|          RH|     Recrutamento Q1|Em Progresso|  Maria Santos|               20|         2500.0|2026-04-22 12:12:...|
|  3|    2024-01-17|      Vendas|     Análise Mercado|   Concluído|Pedro Oliveira|               30|         1500.0|2026-04-22 12:12:...|
|  4|    2024-01-18|  Financeiro|     Auditoria Anual|    Pendente|     Ana Costa|               15|            0.0|2026-04-22 12:12:...|
|  5|    2024-01-19|   Operações|O

In [9]:
import os
print(os.getcwd())


/workspaces/lakehouse-lab/notebooks
